# Week 1 class

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/goodman-imperial/discrete-maths-classes/blob/main/week-1-class.ipynb)

The aim of this class is to implement the Fibonacci algorithm using the following methods, and plot their timings (for $n=1$ to 100, except for the recursive algorithm where you should stop at $n=30$):

* Recursive
* Table
* Keep last two values
* Analytic (if you have time)
* Matrix (if you have time)

## Python

You can use any language you like, but I'll use Python for all my code samples and during the class. I think Python is very simple syntactically so the code samples should be easy enough to follow even if you don't know Python. On the other hand, I would strongly encourage you to learn it as it's an incredibly useful language. Here is a nice tutorial on Python, NumPy (numerical library for efficient array computation) and Jupyter (the notebook interface I'm using here):

* [Python Numpy Jupyter tutorial](https://cs231n.github.io/python-numpy-tutorial/)

Here is a little sample of plotting timings for a function in Python.

In [ ]:
# this first line is just used to make the plots appear nicely in the notebook
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time

def fake_function(n):
    time.sleep(0.001*n*n)
    
def timeit(n):
    start = time.time()
    fake_function(n)
    end = time.time()
    return end-start
    
N = [0, 10, 20, 30, 40, 50]
T = [timeit(n) for n in N]
plt.plot(N, T)
plt.xlabel('n')
plt.ylabel('Time (s)');

## Implementation notes on compiled languages

For the other ones, if you're using a compiled language, they might run so fast that in order to time them you should slow down the addition by adding a 1 microsecond wait. I've included a C++ ``busy_sleep`` function below to do that.

```c++
#include <chrono>
#include <iostream>
#include <fstream>

using namespace std;

// this function busy sleeps for n microseconds
void busy_sleep(int n)
{
    auto start = chrono::steady_clock::now();
    while(chrono::duration_cast<chrono::microseconds>(chrono::steady_clock::now() - start).count()<n) {};
}
```

### Recursion
Time Complexity - O(2^n) [actually closer to O(1.6^n)]  
Space Complexity - O(n)

In [ ]:
def fib_r(n):
    if n<=2:
        return 1

    return fib_r(n-1) + fib_r(n-2)

a = int(input("Enter n to calculate the nth term of the fibonacci sequence: "))
print(fib_r(a))

def timeit2(n):
    start = time.time()
    fake_function(n)
    end = time.time()
    return end-start
    
N = list(range(1, 21))
T = [timeit2(n) for n in N]
plt.plot(N, T)
plt.xlabel('n')
plt.ylabel('Time (s)');

### Pen & Paper Method
Time Complexity - O(n)  
Space Complexity - O(n)

In [ ]:
def fib_pp(n, seen):
    if n in seen:
        return seen[n]

    ans = fib_pp(n-1, seen) + fib_pp(n-2, seen)
    seen[n] = ans
    return ans

def fib_table(n):
    seen = {1: 1, 2: 1}
    return fib_pp(n, seen)

print(fib_table(a))

def timeit3(n):
    start = time.time()
    fib_table(n)
    end = time.time()
    return end-start

N = list(range(1, 101))
T = [timeit3(n) for n in N]
plt.plot(N, T)
plt.xlabel('n')
plt.ylabel('Time (s)');
plt.ylim(0, 5e-4)

### Speeding it Up (Integer method)
Time Complexity - O(n)  
Space Complexity - O(1)

In [ ]:
def fib_int(n):
    if n<=2:
        return 1

    minus1 = 1
    minus2 = 1
    for i in range (2, n):
        ans = minus1 + minus2
        minus2 = minus1
        minus1 = ans

    return ans

print(fib_int(a))

def timeit4(n):
    start = time.time()
    fib_int(n)
    end = time.time()
    return end-start

N = list(range(1, 101))
T = [timeit4(n) for n in N]
plt.plot(N, T)
plt.xlabel('n')
plt.ylabel('Time (s)');
plt.ylim(0, 1e-4)

### Analytical Method

$$
F(n) = \frac{\Phi^n - (1-\Phi)^n}{\sqrt{5}}
$$
Where $$ \Phi = \frac{1+\sqrt{5}}{2} $$

Time and space complexity both SEEM to be O(1), but this is not the case!

We need higher precision for larger values of n, so for a 64-bit floating point arithmetic, it will start giving the wrong value around n = 60

The need for multi-precision arithmetic ends up making this less efficient

In [ ]:
PHI = (1 + 5**0.5)/2

def fib_analytic(n):
    return (PHI**n - (1-PHI)**n)/(5**0.5)

print(fib_analytic(a))

def timeit5(n):
    start = time.time()
    fib_analytic(n)
    end = time.time()
    return end-start

N = list(range(1, 101))
T = [timeit5(n) for n in N]
plt.plot(N, T)
plt.xlabel('n')
plt.ylabel('Time (s)');
plt.ylim(0, 1e-4)

### Matrix Method
Under fixed-size arithmetic assumption:  
Time Complexity - O(log n)  
Space Complexity - O(1)


Let

$$
M = \begin{pmatrix}
1 & 1 \\
1 & 0
\end{pmatrix}
$$

Then raising the matrix to the power $n$ gives:

$$
M^n =
\begin{pmatrix}
F(n+1) & F(n) \\
F(n) & F(n-1)
\end{pmatrix}
$$

using proof by induction

where $F(n)$ is the answer.

Using exponentiation by squaring (such as the example below):

$$
M^8 = ((M^2)^2)^2
$$

This reduces the time complexity of computing $M^n$ to **O(log n)**, allowing us to obtain $F(n)$ efficiently.

In [ ]:
def fib_matrix(n):
    if (n <= 2):
        return 1
    
    M = np.array([[1, 1], [1, 0]])
    M_n = np.eye(2)  #initialise as 2D identity matrix

    while n > 0:
        if n % 2 == 1:
            M_n = M_n @ M

        M = M @ M
        n //= 2

    return M_n[0, 1]

print(fib_matrix(a))

def timeit6(n):
    start = time.time()
    fib_matrix(n)
    end = time.time()
    return end-start

N = list(range(1, 101))
T = [timeit6(n) for n in N]
plt.plot(N, T)
plt.xlabel('n')
plt.ylabel('Time (s)');
plt.ylim(0, 1e-3)

### Breaking Down Assumptions

##### INTEGER METHOD:  
Space complexity is not really O(1)  
We know that $x$ bits can store values up to but not including $$ 2^x $$

Therefore, to store an integer n, the number of bits needed would be the smallest integer greater than $$ \log_2 n $$

Using the same reasoning as above and the approximation that F(n) = $$ 0.45*1.6^n $$ this would take $$ \log_2 (0.45*1.6^n) $$ bits to store, which is equal to $$ \log_2 0.45 + n(\log_2 1.6) $$  
This is approximately proportional to n, indicating:  

space complexity = O(n)  

The time it takes to add 2 integers is also proportional to the number of bits used to store them

This makes time complexity = O(n^2)  
since we carry out these intermediate additions (each taking time proportional to n) an n number of times


##### MATRIX METHOD: 

For the same reasons above, the matrix method actually has  
time complexity O(nlog n),  
because each matrix multiplication alos takes time approximately proportional to n